# BirdCLEF+ 2026 — Phase 5: Pseudo-Labeled Training

Adds ~120,000 **pseudo-labeled** soundscape windows to the training set. These are
windows the Phase 3.5 ensemble predicted with confidence > 0.75. They're noisy
compared to real labels but they're ~80× more soundscape data than Phase 3.5 had.

Training set composition (with REPLICAS=10 on the small labeled-SS pool):

```
35,549  clips (multi-hot from primary + secondary labels)
10,340  labeled soundscape windows (1,034 × 10 — same as Phase 3.5)
~120k   pseudo-labeled windows (NEW; from 04_pseudo_label.ipynb)
```

Mix: ~21% clips, ~6% labeled SS, ~73% pseudo SS. Roughly the opposite balance of
Phase 3.5 (which was clip-heavy). The bet: more target-distribution data >
slightly-noisier labels.

Architecture: same PerchHead MLP + mixup. 5-seed ensemble. Saved as `model_v6_seed*.pt`.

**Prereq**: `04_pseudo_label.ipynb` must have completed (asserts at top).


## 1. Setup


In [ ]:
import os, ast, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, ConcatDataset

from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Device:", DEVICE, "  Torch:", torch.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EMBED_DIR    = PROJECT_ROOT / "embeddings"
CKPT_DIR     = PROJECT_ROOT / "checkpoints"

# Prereqs from prior notebooks
for f in [
    "clip_embeddings.npy", "clip_index.csv",
    "ss_window_embeddings.npy", "ss_window_index.csv",
    "unlabeled_ss_embeddings.npy", "unlabeled_ss_index.csv",
    "pseudo_labels.npy",                                      # from 04_pseudo_label.ipynb
]:
    assert (EMBED_DIR / f).exists(), f"Missing {EMBED_DIR / f}"

SEED      = 42
EMBED_DIM = 1536
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


## 2. Load all embeddings + labels

- Clip embeddings (35,549 × 1536) + clip metadata
- Labeled soundscape embeddings (1,478 × 1536) + labels CSV
- Unlabeled soundscape embeddings (~126k × 1536) + pseudo-label matrix


In [ ]:
# Clips
clip_embeddings = np.load(EMBED_DIR / "clip_embeddings.npy")
clip_index      = pd.read_csv(EMBED_DIR / "clip_index.csv")
clip_index["primary_label"] = clip_index["primary_label"].astype(str)

# Labeled soundscape windows
ss_embeddings = np.load(EMBED_DIR / "ss_window_embeddings.npy")
ss_index      = pd.read_csv(EMBED_DIR / "ss_window_index.csv")

# Pseudo-labeled soundscape windows
pseudo_emb    = np.load(EMBED_DIR / "unlabeled_ss_embeddings.npy")
pseudo_index  = pd.read_csv(EMBED_DIR / "unlabeled_ss_index.csv")
pseudo_labels = np.load(EMBED_DIR / "pseudo_labels.npy")   # (N, 206) multi-hot

print(f"Clips:                  {clip_embeddings.shape}")
print(f"Labeled SS windows:     {ss_embeddings.shape}")
print(f"Pseudo-labeled SS:      {pseudo_emb.shape}  pseudo-labels: {pseudo_labels.shape}")
assert len(pseudo_emb) == len(pseudo_labels) == len(pseudo_index)
print(f"\nPseudo-label stats:")
print(f"  windows with 1+ positives: {(pseudo_labels.sum(axis=1) > 0).sum():,}  "
      f"({(pseudo_labels.sum(axis=1) > 0).mean()*100:.1f}%)")
print(f"  total positive labels:     {int(pseudo_labels.sum()):,}")


## 3. Species mapping (same as Phase 3.5)


In [ ]:
species      = sorted(clip_index["primary_label"].unique())
label_to_idx = {s: i for i, s in enumerate(species)}
NUM_CLASSES  = len(species)
print(f"NUM_CLASSES = {NUM_CLASSES}")
assert pseudo_labels.shape[1] == NUM_CLASSES, "pseudo_labels columns must match NUM_CLASSES"


## 4. File-based train/val split for the labeled soundscapes

Same split as Phase 1+ (deterministic via SEED). The 30% val files are
held out and used as our LB proxy. The other 70% join training.


In [ ]:
unique_files = sorted(ss_index["filename"].unique())
shuffled     = np.random.default_rng(SEED).permutation(unique_files)
n_val_files  = max(1, int(len(shuffled) * 0.3))
val_files    = set(shuffled[:n_val_files])

ss_val_mask   = ss_index["filename"].isin(val_files).to_numpy()
ss_train_mask = ~ss_val_mask

ss_train_emb = ss_embeddings[ss_train_mask]
ss_train_idx = ss_index[ss_train_mask].reset_index(drop=True)
ss_val_emb   = ss_embeddings[ss_val_mask]
ss_val_idx   = ss_index[ss_val_mask].reset_index(drop=True)

print(f"Soundscape files: {len(unique_files)} → "
      f"{len(unique_files)-n_val_files} train / {n_val_files} val")
print(f"Soundscape windows: {len(ss_train_idx)} train / {len(ss_val_idx)} val")


## 5. Datasets

Three dataset classes, all producing `(embedding, multi_hot_label)`:

- **`ClipEmbeddingDataset`** — clips. Labels = primary + secondary species.
- **`SoundscapeEmbeddingDataset`** — labeled soundscape windows. Labels =
  semicolon-separated species in the CSV.
- **`PseudoLabelDataset`** — pseudo-labeled windows. Labels = the precomputed
  multi-hot matrix from `04_pseudo_label.ipynb`.


In [ ]:
def parse_secondary_labels(s):
    if pd.isna(s) or s in ("", "[]"):
        return []
    try:
        return list(ast.literal_eval(s))
    except (ValueError, SyntaxError):
        return []


def clip_label_vec(row, num_classes):
    vec = torch.zeros(num_classes, dtype=torch.float32)
    pl = str(row["primary_label"])
    if pl in label_to_idx:
        vec[label_to_idx[pl]] = 1.0
    for sp in parse_secondary_labels(row.get("secondary_labels", "[]")):
        sp = str(sp)
        if sp in label_to_idx:
            vec[label_to_idx[sp]] = 1.0
    return vec


def ss_label_vec(row, num_classes):
    vec = torch.zeros(num_classes, dtype=torch.float32)
    for sp in str(row["primary_label"]).split(";"):
        sp = sp.strip()
        if sp in label_to_idx:
            vec[label_to_idx[sp]] = 1.0
    return vec


class ClipEmbeddingDataset(Dataset):
    def __init__(self, embeddings, index_df, label_fn):
        self.embeddings = embeddings
        self.index_df   = index_df.reset_index(drop=True)
        self.label_fn   = label_fn

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, i):
        emb   = torch.from_numpy(self.embeddings[i].copy())
        label = self.label_fn(self.index_df.iloc[i], NUM_CLASSES)
        return emb, label


class PseudoLabelDataset(Dataset):
    """Pseudo-labels are already a multi-hot float32 matrix."""
    def __init__(self, embeddings, label_matrix):
        self.embeddings   = embeddings
        self.label_matrix = label_matrix

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, i):
        emb   = torch.from_numpy(self.embeddings[i].copy())
        label = torch.from_numpy(self.label_matrix[i].copy())
        return emb, label


# Smoke test
_ds = ClipEmbeddingDataset(clip_embeddings[:3], clip_index.head(3), clip_label_vec)
_e, _l = _ds[0]
print(f"clip ds sample: emb={tuple(_e.shape)}  label={tuple(_l.shape)}  label sum={_l.sum():.0f}")

_psd = PseudoLabelDataset(pseudo_emb[:3], pseudo_labels[:3])
_e, _l = _psd[0]
print(f"pseudo ds sample: emb={tuple(_e.shape)}  label={tuple(_l.shape)}  label sum={_l.sum():.0f}")


## 6. Mixed DataLoader

```
35,549  clips                       (real labels)
10,340  labeled soundscape windows  (replicated 10× — high quality, distribution-matched)
~120k   pseudo-labeled windows      (single copy — abundant but noisier)
```

Total ~165k. Soundscape share ~79%. The opposite balance from Phase 3.5 — we're
intentionally drowning the model in pseudo-labeled target-distribution data.


In [ ]:
BATCH_SIZE          = 256
NUM_WORKERS         = 0
SOUNDSCAPE_REPLICAS = 10

# Filter clips to species the model knows
clip_keep      = clip_index["primary_label"].isin(label_to_idx).to_numpy()
clip_emb_train = clip_embeddings[clip_keep]
clip_idx_train = clip_index[clip_keep].reset_index(drop=True)

# Build the three sub-datasets
clip_ds   = ClipEmbeddingDataset(clip_emb_train, clip_idx_train, clip_label_vec)
ss_ds     = ClipEmbeddingDataset(ss_train_emb,    ss_train_idx,   ss_label_vec)
ss_rep    = ConcatDataset([ss_ds] * SOUNDSCAPE_REPLICAS)
pseudo_ds = PseudoLabelDataset(pseudo_emb, pseudo_labels)

train_ds = ConcatDataset([clip_ds, ss_rep, pseudo_ds])
train_loader_template = lambda: DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
)

# Val: held-out labeled soundscape windows (unchanged from Phase 3.5)
val_ds     = ClipEmbeddingDataset(ss_val_emb, ss_val_idx, ss_label_vec)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Clip train:        {len(clip_ds):,}")
print(f"Labeled SS train:  {len(ss_ds)} × {SOUNDSCAPE_REPLICAS} = {len(ss_rep):,}")
print(f"Pseudo SS:         {len(pseudo_ds):,}")
print(f"Mixed total:       {len(train_ds):,}")
print(f"  clip share:      {len(clip_ds)/len(train_ds):.1%}")
print(f"  labeled SS:      {len(ss_rep)/len(train_ds):.1%}")
print(f"  pseudo SS:       {len(pseudo_ds)/len(train_ds):.1%}")
print(f"Val:               {len(val_ds):,} windows")


## 7. PerchHead architecture (same as Phase 3.5)


In [ ]:
class PerchHead(nn.Module):
    def __init__(self, embed_dim=1536, hidden_dim=512, num_classes=234, dropout=0.3):
        super().__init__()
        self.norm  = nn.LayerNorm(embed_dim)
        self.drop1 = nn.Dropout(0.2)
        self.fc1   = nn.Linear(embed_dim, hidden_dim)
        self.act   = nn.ReLU(inplace=True)
        self.drop2 = nn.Dropout(dropout)
        self.fc2   = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.norm(x)
        x = self.drop1(x)
        x = self.act(self.fc1(x))
        x = self.drop2(x)
        return self.fc2(x)


## 8. Metric, mixup, train/validate functions


In [ ]:
def macro_auc(y_true, y_pred):
    scores = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col):
            continue
        try:
            scores.append(roc_auc_score(col, y_pred[:, c]))
        except ValueError:
            continue
    return float(np.mean(scores)) if scores else float("nan")


def mixup_batch(emb, label, alpha=0.4, p=0.5):
    if np.random.random() < p:
        lam  = float(np.random.beta(alpha, alpha))
        perm = torch.randperm(emb.size(0), device=emb.device)
        emb   = lam * emb   + (1 - lam) * emb[perm]
        label = lam * label + (1 - lam) * label[perm]
    return emb, label


def train_one_epoch_mixup(model, loader, optimizer, loss_fn, mixup_alpha=0.4, mixup_p=0.5):
    model.train()
    total, n = 0.0, 0
    for emb, label in loader:
        emb, label = emb.to(DEVICE), label.to(DEVICE)
        emb, label = mixup_batch(emb, label, alpha=mixup_alpha, p=mixup_p)
        optimizer.zero_grad()
        logits = model(emb)
        loss   = loss_fn(logits, label)
        loss.backward()
        optimizer.step()
        total += loss.item(); n += 1
    return total / max(1, n)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for emb, label in loader:
        logits = model(emb.to(DEVICE)).cpu().numpy()
        all_logits.append(logits); all_labels.append(label.numpy())
    y_pred = np.concatenate(all_logits)
    y_true = np.concatenate(all_labels)
    y_prob = 1.0 / (1.0 + np.exp(-y_pred))
    return macro_auc(y_true, y_prob)


## 9. Train 5 seeds → save model_v6_seed{42..46}.pt

Same recipe as Phase 3.5's Experiment 5: 5 random seeds, mixup, save-best by val.
**Fewer epochs (15 instead of 30)** because we now have ~3.6× more data per epoch — the
optimizer sees more gradient steps each epoch, so we converge faster.

Each seed: ~5 min wall-clock (4× more data than Phase 3.5's ~1 min). Total ~25 min.


In [ ]:
SEEDS       = [42, 43, 44, 45, 46]
EPOCHS      = 15
LR          = 5e-4
WD          = 1e-4
MIXUP_ALPHA = 0.4
MIXUP_P     = 0.5

loss_fn     = nn.BCEWithLogitsLoss()
all_history = {}
all_best    = {}

for seed in SEEDS:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    head = PerchHead(EMBED_DIM, hidden_dim=512, num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WD)
    train_loader = train_loader_template()

    best_ss_auc = -1.0
    ckpt_path   = CKPT_DIR / f"model_v6_seed{seed}.pt"
    history     = []

    t_seed = time.time()
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch_mixup(head, train_loader, optimizer, loss_fn,
                                           mixup_alpha=MIXUP_ALPHA, mixup_p=MIXUP_P)
        ss_auc = validate(head, val_loader)
        history.append((epoch, train_loss, ss_auc))

        if ss_auc > best_ss_auc:
            best_ss_auc = ss_auc
            torch.save({
                "state_dict":   head.state_dict(),
                "species":      species,
                "label_to_idx": label_to_idx,
                "num_classes":  NUM_CLASSES,
                "embed_dim":    EMBED_DIM,
                "head_config":  {"hidden_dim": 512, "dropout": 0.3},
                "seed":         seed,
                "phase":        "5_pseudo_labels",
            }, ckpt_path)

    dt = time.time() - t_seed
    print(f"Seed {seed}:  best ss_val_auc={best_ss_auc:.4f}  ({dt:.1f}s, {len(history)} epochs)")
    all_history[seed] = history
    all_best[seed]    = best_ss_auc

print(f"\n=== Per-seed summary ===")
for s, b in all_best.items():
    print(f"  Seed {s}: {b:.4f}")
mean_best = np.mean(list(all_best.values()))
std_best  = np.std(list(all_best.values()))
print(f"  Mean: {mean_best:.4f}   Std: {std_best:.4f}")


## 10. Ensemble evaluation


In [ ]:
def load_head(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    h = PerchHead(EMBED_DIM, **ckpt["head_config"], num_classes=NUM_CLASSES).to(DEVICE)
    h.load_state_dict(ckpt["state_dict"])
    h.eval()
    return h


heads = [load_head(CKPT_DIR / f"model_v6_seed{s}.pt") for s in SEEDS]

all_logits = None
all_labels_concat = []
with torch.no_grad():
    for emb, label in val_loader:
        emb = emb.to(DEVICE)
        batch_logits = sum(h(emb) for h in heads) / len(heads)
        if all_logits is None:
            all_logits = batch_logits.cpu().numpy()
        else:
            all_logits = np.concatenate([all_logits, batch_logits.cpu().numpy()])
        all_labels_concat.append(label.numpy())

y_true = np.concatenate(all_labels_concat)
y_prob = 1.0 / (1.0 + np.exp(-all_logits))
ensemble_auc = macro_auc(y_true, y_prob)

print(f"Phase 5 ensemble ss_val_auc: {ensemble_auc:.4f}")
print()
print(f"Reference points:")
print(f"  Phase 3 baseline (single head):    0.9276   LB 0.836")
print(f"  Phase 3.5 ensemble:                 0.9329   LB 0.833")
print(f"  Phase 5 mean individual:            {mean_best:.4f}")
print(f"  Phase 5 ensemble:                   {ensemble_auc:.4f}   Δ vs Phase 3.5: {ensemble_auc-0.9329:+.4f}")


## 11. Plot the per-seed training curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for seed, hist in all_history.items():
    ep, loss, auc = zip(*hist)
    axes[0].plot(ep, loss, alpha=0.6, label=f"seed {seed}")
    axes[1].plot(ep, auc,  alpha=0.6, label=f"seed {seed}")
axes[0].set_title("train loss (per seed)"); axes[0].set_xlabel("epoch"); axes[0].legend(fontsize=8)
axes[1].set_title("soundscape val_auc (per seed)"); axes[1].set_xlabel("epoch")
axes[1].axhline(0.9329, color="b", linestyle="--", label="Phase 3.5 ensemble (0.9329)")
axes[1].axhline(ensemble_auc, color="purple", linestyle="-", linewidth=2,
                label=f"Phase 5 ensemble {ensemble_auc:.4f}")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


## What's next

If `ensemble_auc > 0.9329 + 0.01` (above noise floor): ship to Kaggle.

I'll then:
1. Upload all 5 `model_v6_seed*.pt` files to the Kaggle dataset.
2. Update the kernel to load the new ensemble.
3. Submit.

Expected LB: **0.86-0.88** if pseudo-labeling delivers its typical +0.03 to +0.05.
